# Exercícios: Cadeias de Markov e MDP

Neste notebook você vai praticar os conceitos vistos em aula:

- Construir e interpretar uma **Matriz de Transição**
- Simular trajetórias em uma **Cadeia de Markov**
- Calcular a **Equação de Bellman** manualmente
- Implementar o algoritmo de **Value Iteration**
- Analisar o impacto do fator de desconto **γ (gamma)** e do movimento ruidoso na grade 4×3 da aula

Cada exercício tem um espaço `# SEU CÓDIGO AQUI`.

### ⏱️ Roteiro (~35 min no núcleo)

| | Exercícios | Tempo |
|---|---|---|
| ⭐ **Essenciais** | Parte 1: **1, 2, 3** · Parte 2: **6, 7, 8, 11** | ~35 min |
| ➕ **Opcionais** (se sobrar tempo ou em casa) | 4 (gráfico das trajetórias) · 9 e 10 (convergência completa e curva) · 12 (ruído + fogo −10) | ~25 min |

O item 5 não é exercício: só **rode** a célula que cria a grade.

Em cada exercício há **duas dicas**: a **Dica 1** dá a ideia; a **Dica 2** mostra o código. Tente com a primeira antes de abrir a segunda.


---

**Conceitos-chave:**

| Símbolo | Nome | Significado |
|---------|------|-------------|
| `P` | Matriz de Transição | Probabilidade de ir do estado `i` para o estado `j` |
| `V(s)` | Função de Valor | Recompensa esperada acumulada a partir do estado `s` |
| `γ` | Gamma | Quanto o agente valoriza recompensas futuras (0 a 1) |
| `δ` | Delta | Variação máxima em V entre iterações (critério de parada) |

## Importações — execute antes de tudo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---
# Parte 1 — Cadeias de Markov

Vamos modelar o comportamento de um **estudante** ao longo de uma semana.
Em cada dia, ele pode estar em um de três estados:

- **Estado 0**: Estudando 📚
- **Estado 1**: Descansando 🛋️
- **Estado 2**: Jogando videogame 🎮

---
## Exercício 1 — Construindo a Matriz de Transição ⭐

Complete a matriz de transição `P` com as seguintes regras de comportamento do estudante:

| De \ Para | Estudando | Descansando | Jogando |
|-----------|-----------|-------------|--------|
| **Estudando** | 0.5 | 0.3 | 0.2 |
| **Descansando** | 0.4 | 0.4 | 0.2 |
| **Jogando** | ??? | 0.3 | 0.5 |

**Regra obrigatória:** cada linha deve somar **exatamente 1.0** (probabilidades totais).

Descubra o valor `???` e complete a linha do estado **Jogando**.

In [ ]:
estados = ['📚 Estudando', '🛋️ Descansando', '🎮 Jogando']

P = np.array([
    [0.5, 0.3, 0.2],   # De Estudando para: Estudando, Descansando, Jogando
    [0.4, 0.4, 0.2],   # De Descansando para: Estudando, Descansando, Jogando
    [???, 0.3, 0.5]    # De Jogando para: Estudando, Descansando, Jogando
                       # SEU CÓDIGO AQUI: substitua ??? pelo valor correto
])

# Verificação automática:
print("Soma de cada linha (deve ser 1.0):")
for i, estado in enumerate(estados):
    soma = P[i].sum()
    ok = "✅" if abs(soma - 1.0) < 1e-9 else "❌"
    print(f"  {estado}: {soma:.2f} {ok}")

<details><summary>💡 Dica 1 — a ideia</summary>

Cada linha da matriz é uma distribuição de probabilidade: precisa somar **1**. Quanto falta para a linha do Jogando chegar a 1?
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

A linha do estado Jogando deve somar 1.0:
```
??? + 0.3 + 0.5 = 1.0
??? = 1.0 - 0.3 - 0.5
```
</details>

---
## Exercício 2 — Simulando uma Trajetória ⭐

Implemente a função `simular_cadeia(P, estado_inicial, num_passos)` que:

1. Começa no `estado_inicial`
2. A cada passo, sorteia o próximo estado com base em `P[estado_atual]`
3. Retorna a lista de estados visitados (incluindo o inicial)

> **Dica:** use `np.random.choice(n_estados, p=P[estado_atual])` para sortear o próximo estado.

In [ ]:
def simular_cadeia(P, estado_inicial=0, num_passos=7):
    """
    Simula uma trajetória na Cadeia de Markov.

    Parâmetros:
        P             : matriz de transição
        estado_inicial: índice do estado de partida
        num_passos    : quantidade de transições a realizar

    Retorna:
        trajetoria: lista de índices de estados visitados
    """
    trajetoria = [estado_inicial]
    estado_atual = estado_inicial

    for _ in range(num_passos):
        # SEU CÓDIGO AQUI:
        # 1. Sorteie o próximo estado usando P[estado_atual]
        # 2. Adicione-o à trajetória
        # 3. Atualize estado_atual
        pass

    return trajetoria


# Teste: simule 3 trajetórias de 7 dias
print("3 simulações de 7 dias, partindo de 📚 Estudando:\n")
for i in range(3):
    traj = simular_cadeia(P, estado_inicial=0, num_passos=7)
    sequencia = " → ".join([estados[s] for s in traj])
    print(f"Simulação {i+1}: {sequencia}")

<details><summary>💡 Dica 1 — a ideia</summary>

Em cada passo, o próximo estado é **sorteado** usando a linha do estado atual como probabilidades. `np.random.choice` aceita um parâmetro `p` com essas probabilidades.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
proximo_estado = np.random.choice(len(P), p=P[estado_atual])
trajetoria.append(proximo_estado)
estado_atual = proximo_estado
```
</details>

---
## Exercício 3 — Frequência dos Estados ⭐

Execute **500 simulações** de 30 passos cada (partindo do estado 0) e calcule:

1. Quantas vezes cada estado apareceu no total
2. A frequência relativa (%) de cada estado
3. Exiba um gráfico de barras com os resultados

> Isso estima a **distribuição estacionária** da cadeia — o comportamento de longo prazo.

In [ ]:
n_simulacoes = 500
n_passos = 30

# Contador de ocorrências
contagem = [0, 0, 0]

for _ in range(n_simulacoes):
    # SEU CÓDIGO AQUI:
    # 1. Simule uma trajetória com simular_cadeia
    # 2. Para cada estado na trajetória, incremente contagem[estado]
    pass

total = sum(contagem)

print("Distribuição estimada dos estados (longo prazo):")
print("-" * 45)
for i, estado in enumerate(estados):
    pct = contagem[i] / total * 100
    barra = "█" * int(pct / 2)
    print(f"{estado:20s}: {pct:5.1f}% {barra}")

# Gráfico de barras
cores = ['#4CAF50', '#2196F3', '#FF9800']
frequencias = [c / total * 100 for c in contagem]

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(estados, frequencias, color=cores, edgecolor='black', linewidth=1.2)

for barra, pct in zip(barras, frequencias):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Frequência (%)', fontsize=11)
ax.set_title('Distribuição Estimada dos Estados (500 simulações)', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(frequencias) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

<details><summary>💡 Dica 1 — a ideia</summary>

Simule muitas trajetórias e **conte** quantas vezes cada estado aparece. A frequência de cada um é a estimativa do longo prazo.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
for _ in range(n_simulacoes):
    traj = simular_cadeia(P, estado_inicial=0, num_passos=n_passos)
    for estado in traj:
        contagem[estado] += 1
```
</details>

---
## Exercício 4 — Visualizando Trajetórias (➕ opcional)

Gere **3 trajetórias** de 10 passos e plote-as em um gráfico de linhas, onde:
- O eixo X representa o dia (passo)
- O eixo Y representa o estado (0, 1 ou 2)
- Cada trajetória é uma linha diferente
- O eixo Y deve mostrar os **nomes dos estados**, não os números

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for i in range(3):
    # SEU CÓDIGO AQUI:
    # 1. Simule uma trajetória de 10 passos
    # 2. Plote com ax.plot(dias, trajetória, 'o-', label=...)
    pass

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(estados, fontsize=10)
ax.set_xlabel('Dia', fontsize=11)
ax.set_title('Trajetórias do Estudante ao Longo do Tempo', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 2.5)
plt.tight_layout()
plt.show()

<details><summary>💡 Dica 1 — a ideia</summary>

Para cada trajetória, o eixo x são os dias (0, 1, 2, …) e o eixo y é o estado em cada dia. Um `ax.plot` por trajetória.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
for i in range(3):
    traj = simular_cadeia(P, estado_inicial=0, num_passos=10)
    dias = range(len(traj))
    ax.plot(dias, traj, 'o-', linewidth=2, markersize=7,
            label=f'Simulação {i+1}', alpha=0.8)
```
</details>

---
# Parte 2 — MDP e Equação de Bellman

Agora vamos trabalhar com a **grade 4×3 da aula** (slides 50 a 66):

```
 ┌───────┬───────┬───────┬───────┐
 │       │       │       │  +1   │   linha 0
 ├───────┼───────┼───────┼───────┤
 │       │ ████  │       │  −1   │   linha 1
 ├───────┼───────┼───────┼───────┤
 │ START │       │       │       │   linha 2
 └───────┴───────┴───────┴───────┘
```

- Cada célula é `(linha, coluna)`, com a **linha 0 no topo**.
- **Diamante (0,3):** +1 ao entrar, e o episódio acaba.
- **Fogo (1,3):** −1 ao entrar, e o episódio acaba.
- **Parede (1,1):** bloqueia; bater nela (ou na borda) faz o agente ficar parado.
- **Demais passos:** recompensa 0.
- **Movimento:** determinístico (`ruido = 0`) ou ruidoso (`ruido = 0.2`: 80% para onde pediu, 10% para cada lado).

---
## Preparação — o ambiente MDP (só rode)

Execute a célula abaixo para criar a grade, as transições e a função de visualização.
Pergunta rápida (opcional): **quantos estados NÃO terminais existem?** (todas as células, menos a parede, o diamante e o fogo)

In [ ]:
LINHAS, COLUNAS = 3, 4
PAREDE, DIAMANTE, FOGO, START = (1, 1), (0, 3), (1, 3), (2, 0)
TERMINAIS = [DIAMANTE, FOGO]

ACOES = ['cima', 'direita', 'baixo', 'esquerda']
DELTA = {'cima': (-1, 0), 'baixo': (1, 0), 'esquerda': (0, -1), 'direita': (0, 1)}
LADOS = {'cima': ('esquerda', 'direita'), 'baixo': ('esquerda', 'direita'),
         'esquerda': ('cima', 'baixo'), 'direita': ('cima', 'baixo')}
SETAS = {'cima': '↑', 'baixo': '↓', 'esquerda': '←', 'direita': '→'}

ESTADOS = [(r, c) for r in range(LINHAS) for c in range(COLUNAS) if (r, c) != PAREDE]


def mover(s, a):
    '''Próxima célula ao executar `a` em `s` (fica parado se bater na parede ou na borda).'''
    r, c = s[0] + DELTA[a][0], s[1] + DELTA[a][1]
    if not (0 <= r < LINHAS and 0 <= c < COLUNAS) or (r, c) == PAREDE:
        return s
    return (r, c)


def transicoes(s, a, ruido):
    '''P(s'|s,a) como lista de (probabilidade, próximo estado).'''
    if ruido == 0:
        return [(1.0, mover(s, a))]
    lado1, lado2 = LADOS[a]
    return [(1 - ruido, mover(s, a)), (ruido / 2, mover(s, lado1)), (ruido / 2, mover(s, lado2))]


def plot_grade(V=None, politica=None, titulo='', ax=None, r_fogo=-1.0, destaque=()):
    '''Desenha a grade com V(s) e as setas da política.'''
    criou = ax is None
    if criou:
        fig, ax = plt.subplots(figsize=(6, 4.5))
    for r in range(LINHAS):
        for c in range(COLUNAS):
            s, x, y = (r, c), c, LINHAS - 1 - r
            if s == PAREDE:
                cor = '#9E9E9E'
            elif s == DIAMANTE:
                cor = '#43A047'
            elif s == FOGO:
                cor = '#E53935'
            else:
                v = 0.0 if V is None else V[s]
                cor = plt.cm.RdYlGn(0.5 + 0.35 * np.clip(v, -1, 1))
            ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=cor, edgecolor='white', linewidth=3))
            if s == DIAMANTE:
                ax.text(x + .5, y + .5, '+1', ha='center', va='center', fontsize=16, color='white', fontweight='bold')
            elif s == FOGO:
                ax.text(x + .5, y + .5, f'{r_fogo:+g}', ha='center', va='center', fontsize=16, color='white', fontweight='bold')
            elif s != PAREDE and V is not None:
                ax.text(x + .5, y + .38, f'{V[s]:.2f}', ha='center', va='center', fontsize=11, fontweight='bold')
                if politica:
                    ax.text(x + .5, y + .7, SETAS[politica[s]], ha='center', va='center', fontsize=18)
            if s == START:
                ax.text(x + .06, y + .06, 'START', fontsize=8, fontweight='bold')
            if s in destaque:
                ax.add_patch(plt.Rectangle((x + .04, y + .04), .92, .92, fill=False, edgecolor='#FF6F00', linewidth=3.5))
    ax.set_xlim(0, COLUNAS)
    ax.set_ylim(0, LINHAS)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    if criou:
        plt.tight_layout()
        plt.show()


plot_grade({s: 0.0 for s in ESTADOS}, titulo='Grade 4×3 — todos V = 0')
print("Teste das transições: estado (2, 2), ação 'cima', ruído 0.2:")
print('  ', transicoes((2, 2), 'cima', 0.2))

In [ ]:
# SEU CÓDIGO AQUI: imprima o número de estados NÃO terminais
print(f"Estados (sem a parede): {len(ESTADOS)}")
print(f"Terminais:              {len(TERMINAIS)}")
print(f"Não terminais:          ???")  # substitua ??? pela expressão correta

<details><summary>💡 Dica 1 — a ideia</summary>

Conte os estados da lista `ESTADOS` que **não** estão em `TERMINAIS`.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
print(f"Não terminais: {len([s for s in ESTADOS if s not in TERMINAIS])}")   # 9
```
</details>

---
## Exercício 6 — Recompensa e Equação de Bellman ⭐

Complete duas funções:

**`recompensa(s_prox, r_fogo)`** — o que o agente ganha ao **entrar** em `s_prox`: +1 no diamante, `r_fogo` no fogo, 0 nos demais.

**`calcular_valor(estado, acao, V, gamma, ruido, r_fogo)`** — o termo dentro do max da equação de Bellman (slide 58):

$$Q(s,a) = \sum_{s'} P(s'|s,a)\,\big[\,R(s') + \gamma \cdot V(s')\,\big]$$

Atenção: se `s'` for terminal, o jogo acaba e o futuro vale **0** (use `V(s') = 0`).

In [ ]:
def recompensa(s_prox, r_fogo=-1.0):
    # SEU CÓDIGO AQUI: +1 no DIAMANTE, r_fogo no FOGO, 0 nos demais
    pass


def calcular_valor(estado, acao, V, gamma=0.9, ruido=0.0, r_fogo=-1.0):
    total = 0.0
    for p, s_prox in transicoes(estado, acao, ruido):
        # SEU CÓDIGO AQUI:
        # 1. futuro = 0 se s_prox for terminal, senão V[s_prox]
        # 2. total += p * (recompensa + gamma * futuro)
        pass
    return total


# Teste com V zerado: só "direita" na casa ao lado do diamante vale 1 (slide 62)
V_teste = {s: 0.0 for s in ESTADOS}
for a in ACOES:
    print(f"  (0,2) {a:9s} → {calcular_valor((0, 2), a, V_teste):.2f}")

<details><summary>💡 Dica 1 — a ideia</summary>

A recompensa é a da casa em que você **entra**. Para o valor da ação: para cada próximo estado possível, some **probabilidade × (recompensa + γ × valor do próximo)**. Se o próximo for terminal, o futuro vale 0.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
def recompensa(s_prox, r_fogo=-1.0):
    if s_prox == DIAMANTE:
        return 1.0
    if s_prox == FOGO:
        return r_fogo
    return 0.0

# dentro do for de calcular_valor:
futuro = 0.0 if s_prox in TERMINAIS else V[s_prox]
total += p * (recompensa(s_prox, r_fogo) + gamma * futuro)
```
</details>

---
## Exercício 7 — Melhor Ação ⭐

Implemente `melhor_acao(estado, V, ...)` que:
1. Testa as 4 ações usando `calcular_valor`
2. Retorna a ação com **maior valor** e o valor correspondente

In [ ]:
def melhor_acao(estado, V, gamma=0.9, ruido=0.0, r_fogo=-1.0):
    melhor_v = -float('inf')
    melhor_a = None

    for acao in ACOES:
        pass  # SEU CÓDIGO AQUI

    return melhor_a, melhor_v


a, v = melhor_acao((0, 2), V_teste)
print(f"Estado (0,2) com V zerado → melhor ação: {a}, valor: {v:.2f}   (esperado: direita, 1.00)")

<details><summary>💡 Dica 1 — a ideia</summary>

Teste as 4 ações com `calcular_valor` e guarde a de **maior** valor — como achar o máximo de uma lista à mão.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
for acao in ACOES:
    v = calcular_valor(estado, acao, V, gamma, ruido, r_fogo)
    if v > melhor_v + 1e-12:   # a tolerância evita trocar de ação por erro de arredondamento
        melhor_v, melhor_a = v, acao
```
</details>

---
## Exercício 8 — Value Iteration (1 iteração manual) ⭐

Implemente **uma varredura** do Value Iteration (γ = 0,9, determinístico):

1. Para cada estado **não terminal**: calcule a melhor ação, guarde o novo valor em `novo_V` e a ação em `politica`
2. Calcule o `delta`: variação máxima entre `V` antigo e novo
3. Substitua `V` por `novo_V`

> Execute a célula **várias vezes**: o valor nasce no diamante e se espalha para trás — 1 → 0,9 → 0,81 → ...

In [ ]:
# Execute esta célula para resetar
V = {s: 0.0 for s in ESTADOS}
iteracao_atual = 0
print("Resetado! Agora execute a célula abaixo repetidamente.")

In [ ]:
# Execute esta célula várias vezes — cada execução é uma varredura
delta = 0
novo_V = dict(V)
politica = {}

for s in ESTADOS:
    if s in TERMINAIS:
        continue
    # SEU CÓDIGO AQUI:
    # 1. acao, valor = melhor_acao(s, V)
    # 2. novo_V[s] = valor  e  politica[s] = acao
    # 3. delta = max(delta, |V[s] - valor|)
    pass

V = novo_V
iteracao_atual += 1
print(f"Iteração {iteracao_atual} | Delta máximo: {delta:.4f}")
plot_grade(V, politica, titulo=f'Iteração {iteracao_atual}')

<details><summary>💡 Dica 1 — a ideia</summary>

Para cada estado **não terminal**: pegue a melhor ação e o valor dela (usando o `V` antigo), guarde em `novo_V` e em `politica`. O `delta` é a **maior** mudança entre o valor antigo e o novo.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
acao, valor = melhor_acao(s, V)
novo_V[s] = valor
politica[s] = acao
delta = max(delta, abs(V[s] - valor))
```
</details>

---
## Exercício 9 — Convergência Completa (➕ opcional)

Implemente o loop completo do Value Iteration até **delta < 1e-6**, guardando cada delta em `historico_delta`.

Depois, compare com os valores da aula (slides 62–64): **1 · 0,9 · 0,81 · 0,73 · 0,66**.

> Pulou este? Tudo bem: o **Exercício 11** pede o mesmo loop dentro de uma função — é o Exercício 8 repetido até o delta ficar pequeno.

In [ ]:
V = {s: 0.0 for s in ESTADOS}
historico_delta = []
iteracao = 0
threshold = 1e-6

while True:
    # SEU CÓDIGO AQUI:
    # faça uma varredura completa (como no Exercício 8),
    # guarde o delta em historico_delta e pare quando delta < threshold
    break  # ← remova este break quando implementar o loop (senão ele roda para sempre)

print(f"Total de iterações: {iteracao}")
aula = {(0, 2): 1.00, (0, 1): 0.90, (0, 0): 0.81, (1, 0): 0.73, (2, 0): 0.66}
for s, v in aula.items():
    print(f"  {s}: aula = {v:.2f} | calculado = {V[s]:.4f}")
plot_grade(V, politica, titulo=f'V* e π* — convergiu em {iteracao} iterações')

<details><summary>💡 Dica 1 — a ideia</summary>

É o Exercício 8 dentro de um `while True`: repita a varredura e pare quando o `delta` ficar menor que o limiar.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
while True:
    delta, novo_V, politica = 0, dict(V), {}
    for s in ESTADOS:
        if s in TERMINAIS:
            continue
        acao, valor = melhor_acao(s, V)
        novo_V[s], politica[s] = valor, acao
        delta = max(delta, abs(V[s] - valor))
    V = novo_V
    iteracao += 1
    historico_delta.append(delta)
    if delta < threshold:
        break
```
</details>

---
## Exercício 10 — Curva de Convergência (➕ opcional)

Plote o `historico_delta` do exercício anterior em **escala logarítmica** (`ax.semilogy`), com uma linha tracejada vermelha no limiar `1e-6`.

> No caso determinístico o delta chega a **exatamente 0** (o log de 0 não existe). Use `np.maximum(historico_delta, 1e-12)` para plotar.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

# SEU CÓDIGO AQUI:
# 1. ax.semilogy(...) com np.maximum(historico_delta, 1e-12)
# 2. ax.axhline(...) no limiar 1e-6

ax.set_xlabel('Iteração')
ax.set_ylabel('Delta máximo (log)')
ax.set_title('Convergência do Value Iteration', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<details><summary>💡 Dica 1 — a ideia</summary>

`ax.semilogy` desenha o eixo y em escala logarítmica; `ax.axhline` desenha uma linha horizontal.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
ax.semilogy(range(1, len(historico_delta) + 1), np.maximum(historico_delta, 1e-12), 'o-', label='Delta')
ax.axhline(1e-6, color='red', linestyle='--', label='Limiar (1e-6)')
ax.legend()
```
</details>

---
## Exercício 11 — O Impacto do γ (Gamma) ⭐

Transforme o loop do Exercício 9 numa função `value_iteration(gamma, ruido=0.0, r_fogo=-1.0)` que devolve `(V, politica, n_iter)`.

Depois rode para **γ ∈ [0.5, 0.7, 0.9, 0.99]** e mostre o valor do START em cada caso.

> **Antes de rodar:** o START está a 4 passos da casa que vale 1. Quanto ele vale com γ = 0,5?

In [ ]:
def value_iteration(gamma, ruido=0.0, r_fogo=-1.0, threshold=1e-6):
    V = {s: 0.0 for s in ESTADOS}
    politica, n_iter = {}, 0
    # SEU CÓDIGO AQUI: o mesmo loop do Exercício 9, passando gamma, ruido e r_fogo para melhor_acao
    return V, politica, n_iter


gammas = [0.5, 0.7, 0.9, 0.99]
fig, axes = plt.subplots(1, len(gammas), figsize=(20, 4))
for ax, g in zip(axes, gammas):
    V_g, pi_g, n = value_iteration(g)
    plot_grade(V_g, pi_g, ax=ax, titulo=f'γ = {g} | V(START) = {V_g[START]:.3f} | {n} iterações')
plt.tight_layout()
plt.show()

<details><summary>💡 Dica 1 — a ideia</summary>

Coloque o loop do Exercício 9 (o Exercício 8 repetido) dentro da função, passando `gamma`, `ruido` e `r_fogo` para `melhor_acao`, e conte as iterações.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
def value_iteration(gamma, ruido=0.0, r_fogo=-1.0, threshold=1e-6):
    V = {s: 0.0 for s in ESTADOS}
    politica, n_iter = {}, 0
    while True:
        delta, novo_V, politica = 0, dict(V), {}
        for s in ESTADOS:
            if s in TERMINAIS:
                continue
            acao, valor = melhor_acao(s, V, gamma, ruido, r_fogo)
            novo_V[s], politica[s] = valor, acao
            delta = max(delta, abs(V[s] - valor))
        V = novo_V
        n_iter += 1
        if delta < threshold:
            return V, politica, n_iter
```
</details>

---
## Exercício 12 — Movimento ruidoso e fogo valendo −10 (slide 66, item 4) (➕ opcional)

Com **ruído 0,2** (80/10/10) e γ = 0,9:

1. Rode `value_iteration` com o fogo valendo −1 e depois −10.
2. Liste as casas em que a política **mudou** e desenhe as duas grades lado a lado, destacando essas casas.

> **Antes de rodar:** aposte quais casas vão trocar de seta.

In [ ]:
V_a, pi_a, _ = value_iteration(0.9, ruido=0.2, r_fogo=-1)
V_b, pi_b, _ = value_iteration(0.9, ruido=0.2, r_fogo=-10)

# SEU CÓDIGO AQUI: lista das casas em que pi_a e pi_b são diferentes
mudaram = ???

print('Casas em que a política mudou:', mudaram)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_grade(V_a, pi_a, ax=axes[0], titulo='Fogo = −1')
plot_grade(V_b, pi_b, ax=axes[1], r_fogo=-10, destaque=mudaram, titulo='Fogo = −10')
plt.tight_layout()
plt.show()

<details><summary>💡 Dica 1 — a ideia</summary>

Percorra as casas da política `pi_a` e guarde as que têm ação **diferente** em `pi_b`.
</details>

<details>
<summary>🔑 Dica 2 — o código</summary>

```python
mudaram = [s for s in pi_a if pi_a[s] != pi_b[s]]   # → [(1, 2), (2, 2), (2, 3)]
```
</details>

---
##  Reflexão Final

Responda as perguntas abaixo (clique duas vezes para editar):

**1. O que acontece com os valores V(s) quando gamma é muito pequeno (ex: 0.5)?**

> *Sua resposta aqui...*

**2. Por que estados mais distantes do objetivo têm valores menores?**

> *Sua resposta aqui...*

**3. Por que, com o fogo valendo −10, algumas casas passam a preferir bater na parede ou se afastar do fogo?**

> *Sua resposta aqui...*

**4. Na Cadeia de Markov do estudante, qual estado você esperaria ser o mais frequente no longo prazo? Isso se confirmou na simulação?**

> *Sua resposta aqui...*

**5. Qual a diferença entre uma Cadeia de Markov (Parte 1) e um MDP (Parte 2)?**

> *Sua resposta aqui...*

---
**Parabéns por concluir os exercícios!**

Você implementou do zero os blocos fundamentais de um MDP:
- Matriz de transição e simulação estocástica
- Equação de Bellman
- Value Iteration com convergência
- Análise do fator de desconto γ e do movimento ruidoso

Na próxima aula veremos como aprender essas mesmas políticas **sem conhecer P(s'|s,a)** — chegando ao **Q-Learning**!